# 4. 실습 데이터 추출 및 분할 

In [2]:
import shutil
import sys

print("Python:", sys.version.split()[0])
print("가상환경 경로:", sys.prefix)
print("가상환경 안에서 실행 중?", ".venv" in sys.prefix)
print()

# OS 패키지 확인 (pip로 설치되지 않는 것들)
print("Poppler  :", "OK" if shutil.which("pdftoppm") else "✗ 미설치 — Path 확인 후 재시작")
print("Tesseract:", "OK" if shutil.which("tesseract") else "✗ 미설치 — Path 확인 후 재시작")
print()

# 파이썬 패키지 확인
for mod in ["langchain", "unstructured", "chromadb", "tiktoken", "PIL"]:
    try:
        __import__(mod)
        print(f"{mod:14s}: OK")
    except ImportError:
        print(f"{mod:14s}: ✗ 미설치")

Python: 3.11.9
가상환경 경로: c:\workspaces\multimodal-rag\.venv
가상환경 안에서 실행 중? True

Poppler  : OK
Tesseract: OK

langchain     : OK
unstructured  : OK
chromadb      : OK
tiktoken      : OK
PIL           : OK


In [3]:
from langchain_text_splitters import CharacterTextSplitter
from unstructured.partition.pdf import partition_pdf


def extract_pdf_elements(path, fname):
    """
    PDF 파일에서 이미지, 테이블, 텍스트 블록을 추출합니다.
    path: 파일 경로 (추출된 이미지 파일이 저장될 위치)
    fname: 파일 이름
    """
    return partition_pdf(
        filename=path + fname,
        strategy="hi_res",                # 레이아웃 분석 모델로 정밀 추출 (교재에 없음! 필수 추가)
        extract_images_in_pdf=True,       # PDF에서 이미지를 추출
        infer_table_structure=True,       # 테이블 구조를 추론
        chunking_strategy="by_title",     # 타이틀을 기준으로 텍스트를 블록으로 분할
        max_characters=4000,              # 최대 4000자로 텍스트 블록을 제한
        new_after_n_chars=3800,           # 3800자 이후에 새로운 블록 생성
        combine_text_under_n_chars=2000,  # 2000자 이하의 텍스트는 결합
        image_output_dir_path=path,       # 이미지가 저장될 경로 설정
    )

In [4]:
def categorize_elements(raw_pdf_elements):
    """
    PDF에서 추출한 요소들을 테이블과 텍스트로 분류합니다.
    raw_pdf_elements: unstructured.documents.elements 리스트
    """
    tables = []
    texts = []
    for element in raw_pdf_elements:
        if "unstructured.documents.elements.Table" in str(type(element)):
            tables.append(str(element))   # 테이블 요소를 저장
        elif "unstructured.documents.elements.CompositeElement" in str(type(element)):
            texts.append(str(element))    # 텍스트 요소를 저장
    return texts, tables

In [5]:
import os

# 현재 작업 디렉터리 경로를 반환합니다.
current_directory = os.getcwd()

# fname: 처리할 PDF 파일 이름 / fpath: PDF가 저장된 폴더 경로
fname = "invest.pdf"
fpath = os.path.join(os.path.dirname(current_directory), "invest/")

print("현재 스크립트의 위치:", current_directory)
print("pdf 위치:", fpath)

# 실행하면 2~3분 걸립니다. 멈춘 게 아니니 기다리세요.
raw_pdf_elements = extract_pdf_elements(fpath, fname)
texts, tables = categorize_elements(raw_pdf_elements)

Thank you for using Unstructured Open Source! Try Unstructured's commercial offering for improved output quality. 10,000 free pages to start. Learn more: https://transform.unstructured.io/?utm_source=unstructured_oss Hide this notice: UNSTRUCTURED_DISABLE_NOTICE=1


현재 스크립트의 위치: c:\workspaces\multimodal-rag\notebooks
pdf 위치: c:\workspaces\multimodal-rag\invest/


No languages specified, defaulting to English.
Loading weights: 100%|██████████| 367/367 [00:00<00:00, 3343.26it/s]


In [12]:
# chunk_size=2000: 각 텍스트 블록의 최대 크기
# chunk_overlap=200: 청크 간 200자의 겹침을 허용해 중요한 정보를 잃지 않도록 함
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=2000, chunk_overlap=200
)

# 텍스트 리스트를 하나의 긴 문자열로 결합한 뒤 2000자 단위로 분할
joined_texts = " ".join(texts)
texts_2k_token = text_splitter.split_text(joined_texts)

print("분할 후 블록 수:", len(texts_2k_token))
print("원본 블록 수:", len(texts))
print(texts[0])
print(type(texts_2k_token))

분할 후 블록 수: 12
원본 블록 수: 18
7} SANS AAS AAs oi7| Alejo BA 1 ro oF

Lead Article

o]Z|ok VA R(LBVAR) ! EES Tet, HES] WS +8 Az}, LBVAR YS — ob AY] 2Aet Sas = aA WHS AZAR St GG)2] BAS Ch, ARES BHO] HSS 7A] & 7] 41%, 0.68% -BLE Al AST AOe PSEC}, ES A] 7A] 4 AGe]oo dst AASB Ao We Aye] 2 Bol tet Ok BO] CSA] UEP. SAA Aa] AUF CWS ASHE Sq Ss 3] 19 BEWA AALS F Ac. aA HS SA AG QWAE 714] BEM 418 oH SB S47} oF FEAT, De} Dee AA QIAS S41 ATS] Hsp} ayers el 20] TA. B ASE AAS2] BEYS AGe]o 7] AS AS AAS aaa z, RAGS AAAAY YAS B7t A AANA GS 4-8 et ASA DAS Asstt.

JEL Classification: C11, C32, C53, G21, G28

HA BAO: Seeks, AAA, Hate BO]Zz/oF VAR, BE WS

DOI: 10.23299/bokeri.2026.32.2.001

* tae Aa Sa AS py, aden Aalst AAp-4y (Email: etpark@bok.or.kr, Tel: 02-759-4114)

WAIAA, Lea Stal 4A] sha} a= (Email: kyuho@korea.ac.kr, Tel! 02-3290-5132) He *

B EEO] Ge] SY EGS IRA VEGA LS, USE, BOI, RSS TWH SE Bo] yy AHA ZARESIUE. B APL Telehahswo) 219K 250773S Wor FIISUE B AHS WSS AAS] MA Yael, HLYO] Ba Adpeke HABYc}, B

# 5 실습 ② 텍스트·표 요약 

In [13]:
import os
from dotenv import load_dotenv

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")

if not openai_api_key:
    raise ValueError("OpenAI API 키가 없습니다. 한번 더 확인 부탁드립니다.")
else:
    print("잘했어요 친구")

잘했어요 친구


In [14]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_ollama.llms import OllamaLLM


def generate_text_summaries(texts, tables, summarize_texts=False):
    """
    텍스트 및 표 데이터를 요약하여 검색에 활용할 수 있는 요약본을 생성합니다.
    texts: 텍스트 리스트
    tables: 표 리스트
    summarize_texts: 텍스트 요약을 활성화할지 여부를 결정하는 불리언 값
    """

    # Prompt 한국어 버전
    prompt_text_kor = """당신은 표와 텍스트를 요약하여 검색에 활용할 수 있도록 돕는 도우미입니다.
    이 요약본들은 임베딩되어 원본 텍스트나 표 요소를 검색하는 데 사용될 것입니다.
    주어진 표나 텍스트의 내용을 검색에 최적화된 간결한 요약으로 작성해주세요.
    이 문서는 현대엘리베이터 N:EX PLUS 제품 카탈로그다. 
    입력이 깨졌거나 판독 불가이면 추측하지 말고 UNREADABLE만 출력하라
    요약할 표 또는 텍스트: {element}"""

    prompt = ChatPromptTemplate.from_template(prompt_text_kor)

    # 모델: GPT-4o-mini or Llama3.1 모델 사용
    # llamaModel = OllamaLLM(model="llama3.1:8b")
    model = ChatOpenAI(temperature=0, model="gpt-4o-mini")

    # 요약 체인 = 프롬프트 + LLM + 출력 파서
    summarize_chain = {"element": lambda x: x} | prompt | model | StrOutputParser()

    text_summaries = []
    table_summaries = []

    # 텍스트 요약을 활성화한 경우
    if texts and summarize_texts:
        text_summaries = summarize_chain.batch(texts, {"max_concurrency": 5})
    # 텍스트 요약을 사용하지 않는 경우
    elif texts:
        text_summaries = texts

    # 테이블 데이터 요약
    if tables:
        table_summaries = summarize_chain.batch(tables, {"max_concurrency": 5})

    return text_summaries, table_summaries


text_summaries, table_summaries = generate_text_summaries(
    texts_2k_token, tables, summarize_texts=True
)
print(text_summaries)
print(table_summaries)

"""
max_concurrency=5는 한 번에 5개씩 동시에 요약하라는 뜻입니다. 숫자를 올리면 빨라지지만 API 요청 제한에 걸릴 수 있습니다. """

['UNREADABLE', 'UNREADABLE', 'UNREADABLE', 'UNREADABLE', 'UNREADABLE', 'UNREADABLE', 'UNREADABLE', 'UNREADABLE', 'UNREADABLE', 'UNREADABLE', 'UNREADABLE', 'UNREADABLE']
['UNREADABLE', '| Horizon (H) | MAE (%) | RMSE (%) | Forecast Household | Corporate Forecast Household |\n|-------------|---------|----------|---------------------|------------------------------|\n| 1           | 0.02    | 0.04     | 1                   | 0.02                         | 0.05                         |\n| 2           | 0.03    | 0.05     | 2                   | 0.04                         | 0.06                         |\n| 3           | 0.04    | 0.07     | 3                   | 0.06                         | 0.08                         |\n| 4           | 0.05    | 0.09     | 4                   | 0.07                         | 0.10                         |\n| **Average** | 0.03    | 0.06     |                     | **Average** 0.05             | 0.07                         |\n\n요약: MAE와 RMSE 값은 각각 1에

'\nmax_concurrency=5는 한 번에 5개씩 동시에 요약하라는 뜻입니다. 숫자를 올리면 빨라지지만 API 요청 제한에 걸릴 수 있습니다. '

# 6 실습 ③ 이미지 요약

In [15]:
import base64
import os
import requests
import json
from langchain_core.messages import HumanMessage


def encode_image(image_path):
    """이미지를 base64 문자열로 변환"""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

In [16]:
def image_summarize(img_base64, prompt):
    """이미지 요약 생성"""
    # ❌ 교재: model="gpt-4-vision-preview" → 지원 종료(deprecated)
    # ✅ 현재: gpt-4o-mini 가 비전 기능을 포함하며 비용도 훨씬 저렴합니다
    chat = ChatOpenAI(model="gpt-4o-mini", max_tokens=1024)

    msg = chat.invoke(
        [
            HumanMessage(
                content=[
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        # Base64 문자열 앞에 'data:image/jpeg;base64,' 접두사가 꼭 필요합니다
                        "image_url": {"url": f"data:image/jpeg;base64,{img_base64}"},
                    },
                ]
            )
        ]
    )
    return msg.content

In [17]:
def generate_img_summaries(path):
    """
    이미지의 요약과 Base64 인코딩 문자열을 생성
    path: Unstructured에 의해 추출된 .jpg 파일의 경로
    """
    img_base64_list = []   # Base64로 인코딩된 이미지를 저장할 리스트
    image_summaries = []   # 이미지 요약을 저장할 리스트

    # Prompt_kor 한국어
    prompt_kor = """You are an assistant tasked with summarizing images for retrieval.
    These summaries will be embedded and used to retrieve the raw image.
    Provide a concise summary of the image that is well optimized for retrieval.
    The summary should be written in Korean (Hangul)."""

    # Prompt 영어
    prompt = """You are an assistant tasked with summarizing images for retrieval.
    These summaries will be embedded and used to retrieve the raw image.
    Provide a concise summary of the image that is well optimized for retrieval."""

    # 주어진 경로에서 파일 목록을 가져와 정렬한 후, 각 파일을 처리합니다.
    for img_file in sorted(os.listdir(path)):
        # 파일이 .jpg, .png, .jpeg 확장자 중 하나일 경우에만 처리합니다.
        if img_file.endswith((".jpg", ".png", ".jpeg")):
            img_path = os.path.join(path, img_file)
            base64_image = encode_image(img_path)
            img_base64_list.append(base64_image)
            image_summaries.append(image_summarize(base64_image, prompt_kor))

    return img_base64_list, image_summaries


img_base64_list, image_summaries = generate_img_summaries(fpath + "figures/")
print(image_summaries[0])

FileNotFoundError: [WinError 3] 지정된 경로를 찾을 수 없습니다: 'c:\\workspaces\\multimodal-rag\\invest/figures/'

# 7. 실습 ④ 다중 벡터 검색기와 최종 체인

In [ ]:
import uuid
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

# 요약문이 들어갈 벡터 DB (= 색인 카드 서랍)
vectorstore = Chroma(
    collection_name="multimodal_rag",
    embedding_function=OpenAIEmbeddings(),
)

# 원본이 들어갈 저장소 (= 서가)
store = InMemoryStore()
id_key = "doc_id"

retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=store,
    id_key=id_key,
)

In [ ]:
def add_documents(retriever, doc_summaries, doc_contents):
    """요약본은 벡터 DB에, 원본은 docstore에 '같은 id'로 저장합니다."""
    doc_ids = [str(uuid.uuid4()) for _ in doc_contents]

    summary_docs = [
        Document(page_content=s, metadata={id_key: doc_ids[i]})
        for i, s in enumerate(doc_summaries)
    ]

    retriever.vectorstore.add_documents(summary_docs)            # 요약 → 검색용
    retriever.docstore.mset(list(zip(doc_ids, doc_contents)))    # 원본 → 전달용


if text_summaries:
    add_documents(retriever, text_summaries, texts_2k_token)
if table_summaries:
    add_documents(retriever, table_summaries, tables)
if image_summaries:
    add_documents(retriever, image_summaries, img_base64_list)

In [ ]:
import re


def split_image_text_types(docs):
    """검색 결과를 Base64 이미지와 텍스트로 나눕니다."""
    b64_images = []
    texts = []
    for doc in docs:
        doc = doc.page_content if isinstance(doc, Document) else doc
        # Base64 문자열은 영문/숫자/+/= 만으로 길게 이어집니다
        if re.match("^[A-Za-z0-9+/]{100,}={0,2}$", doc):
            b64_images.append(doc)
        else:
            texts.append(doc)
    return {"images": b64_images, "texts": texts}

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough


def img_prompt_func(data_dict):
    """검색된 원본을 멀티모달 LLM이 읽을 형태로 조립합니다."""
    messages = []

    # 1) 이미지를 먼저 넣습니다
    for image in data_dict["context"]["images"]:
        messages.append(
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image}"},
            }
        )

    # 2) 텍스트 컨텍스트와 질문을 넣습니다
    formatted_texts = "\n".join(data_dict["context"]["texts"])
    messages.append(
        {
            "type": "text",
            "text": (
                "아래 자료를 근거로 질문에 한국어로 답하세요.\n"
                "자료에 없는 내용은 모른다고 답하세요.\n\n"
                f"질문: {data_dict['question']}\n\n"
                f"참고 텍스트:\n{formatted_texts}"
            ),
        }
    )
    return [HumanMessage(content=messages)]


model = ChatOpenAI(temperature=0, model="gpt-4o-mini", max_tokens=1024)

chain = (
    {
        "context": retriever | RunnableLambda(split_image_text_types),
        "question": RunnablePassthrough(),
    }
    | RunnableLambda(img_prompt_func)
    | model
    | StrOutputParser()
)

query = "2023년 하반기 매출 추세를 그래프 기준으로 설명해줘"
print(chain.invoke(query))

In [ ]:
translate_prompt = ChatPromptTemplate.from_template(
    "다음 영문 답변을 자연스러운 한국어로 번역하세요. 숫자와 고유명사는 그대로 두세요.\n\n{answer}"
)

# 번역은 로컬 오픈소스 모델로 처리해 비용을 줄일 수 있습니다
translator = OllamaLLM(model="llama3.1:8b")

final_chain = chain | {"answer": RunnablePassthrough()} | translate_prompt | translator | StrOutputParser()
print(final_chain.invoke(query))